In [9]:
import torch
import torch.nn.functional as F

T4_PEAK_BW_GBS = 320.0     # T4 memory bandwidth peak
T4_PEAK_GFLOPS = 8141.0    # T4 FP32 compute peak

In [10]:
!pip install ninja

In [11]:
# ------------------------------------------------------------
# Standard central-difference coefficients for the 1D second
# derivative (Fornberg), order 2/4/6 <-> radius 1/2/3.
# Indexed [0]=center, [1]=+-1, [2]=+-2, [3]=+-3.
# ------------------------------------------------------------
COEFFS_1D = {
    1: [-2.0, 1.0],                                  # 2nd order
    2: [-5.0 / 2.0, 4.0 / 3.0, -1.0 / 12.0],          # 4th order
    3: [-49.0 / 18.0, 3.0 / 2.0, -3.0 / 20.0, 1.0 / 90.0],  # 6th order
}


def build_coeffs_array(radius: int, dtype=torch.float32, device="cuda") -> torch.Tensor:
    """
    Single source of truth for stencil coefficients — shared by both
    build_stencil_kernel() (the dense conv3d reference, below) and the
    CUDA kernels' `coeffs` argument (see stencil_kernels.cu's
    launch_naive, which expects exactly this layout).

    Returns a 1D tensor of length radius+1:
        [0]    = center coefficient, summed over all 3 axes (3 * c0)
        [1..r] = neighbor coefficient at distance k (same value used
                 for all 3 axes, since the stencil is isotropic)
    """
    c = COEFFS_1D[radius]
    center = 3.0 * c[0]
    values = [center] + list(c[1:])
    return torch.tensor(values, dtype=dtype, device=device)


def build_stencil_kernel(radius: int, dtype=torch.float32, device="cuda"):
    """
    Build a dense (2r+1)^3 kernel tensor for use with F.conv3d that is
    exactly equivalent to the sparse axis-aligned stencil of the given
    radius. All entries are zero except the center and the six
    axis-aligned neighbor rays.

    Returns shape (1, 1, 2r+1, 2r+1, 2r+1) — ready for F.conv3d's
    (out_channels, in_channels, kD, kH, kW) weight layout.
    """
    coeffs = build_coeffs_array(radius, dtype=dtype, device=device)
    size = 2 * radius + 1
    mid = radius
    k = torch.zeros((size, size, size), dtype=dtype, device=device)

    k[mid, mid, mid] = coeffs[0]

    for offset in range(1, radius + 1):
        c = coeffs[offset]
        # z axis (dim 0), y axis (dim 1), x axis (dim 2)
        k[mid + offset, mid, mid] = c
        k[mid - offset, mid, mid] = c
        k[mid, mid + offset, mid] = c
        k[mid, mid - offset, mid] = c
        k[mid, mid, mid + offset] = c
        k[mid, mid, mid - offset] = c

    return k.unsqueeze(0).unsqueeze(0)  # (1, 1, size, size, size)


def run_torch_reference(inp: torch.Tensor, radius: int) -> torch.Tensor:
    """
    inp: (D, H, W) float32 CUDA tensor.
    Returns (out_D, out_H, out_W) = (D-2r, H-2r, W-2r) — valid mode,
    matching the CUDA kernels' boundary convention (no padding).
    """
    kernel = build_stencil_kernel(radius, dtype=inp.dtype, device=inp.device)
    out = F.conv3d(
        inp.unsqueeze(0).unsqueeze(0),   # (1, 1, D, H, W)
        kernel,
        padding=0,
    )
    return out.squeeze(0).squeeze(0)

In [12]:
# ------------------------------------------------------------
# Timing
# ------------------------------------------------------------
def gpu_timer(fn, *args, warmup=3, runs=20):
    """
    Time a GPU function with CUDA events.
    warmup runs are discarded (primes L2, triggers clock boost).
    Returns average milliseconds over `runs` iterations.
    """
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(runs):
        fn(*args)
    end.record()
    torch.cuda.synchronize()

    return start.elapsed_time(end) / runs


def compute_metrics(N: int, radius: int, time_ms: float):
    """
    Effective bandwidth for an N^3 cubic grid, valid-mode output.
    bytes_moved counts each input/output point once (see problem.md
    "Performance Metrics" — this is a *minimum bytes* convention,
    not what any particular kernel actually fetches from DRAM).

    flops assumes exactly 6*radius+1 taps per output point — true
    for your own kernels (they only ever touch the nonzero taps),
    but NOT true for run_torch_reference(), which computes the full
    dense (2r+1)^3 cube under the hood. Only call this on timings
    from your own launch_* functions, not on the torch reference.
    """
    out_n = N - 2 * radius
    in_pts = N ** 3
    out_pts = max(out_n, 0) ** 3
    bytes_moved = (in_pts + out_pts) * 4  # float32
    seconds = time_ms / 1000.0
    bw_gbs = bytes_moved / seconds / 1e9
    flops = 2 * (6 * radius + 1) * out_pts
    gflops = flops / seconds / 1e9
    return {
        "N": N,
        "radius": radius,
        "time_ms": time_ms,
        "bandwidth_GBs": bw_gbs,
        "pct_peak_bw": bw_gbs / T4_PEAK_BW_GBS * 100.0,
        "gflops": gflops,
    }

In [13]:
# Testing the pytorch reference
for radius in (1, 2, 3):
  N = 16
  inp = torch.randn(N, N, N, device="cuda", dtype=torch.float32)
  out = run_torch_reference(inp, radius)
  expected_shape = (N - 2 * radius,) * 3
  assert tuple(out.shape) == expected_shape, (out.shape, expected_shape)
  assert torch.isfinite(out).all()
  print(f"radius={radius}: output shape {tuple(out.shape)} OK")


radius=1: output shape (14, 14, 14) OK
radius=2: output shape (12, 12, 12) OK
radius=3: output shape (10, 10, 10) OK


In [14]:
%%writefile stencil_kernels.cu
#include <torch/extension.h>
#define MAXR 3
__constant__ float d_coeffs[MAXR+1];
// flat index helper
__device__ __forceinline__ size_t idx3d(int d, int r, int c, int H, int W)
{
    // Explicitly cast all dimensions to size_t to prevent potential intermediate integer overflow
    // when H or W are large, ensuring correct index calculation.
    return static_cast<size_t>(d) * static_cast<size_t>(H) * static_cast<size_t>(W) +
           static_cast<size_t>(r) * static_cast<size_t>(W) +
           static_cast<size_t>(c);
}

__device__ __forceinline__ int smem_idx(int i, int j, int k, int in_tile)
{
    return (i * in_tile + j) * in_tile + k;
}

__device__ __forceinline__ int plane_idx(int r, int c, int in_tile)
{
  return r * in_tile + c;
}

// ============================================================
// VERSION 1: GPU NAIVE
// One thread per output point. Reads its 6*radius+1 neighbors
// directly from global memory; coefficients passed as a small
// device array (length radius+1: [0]=center, [1..radius]=taps).
// ============================================================
__global__ void stencil_naive_kernel(
    const float* __restrict__ input,
    const float* __restrict__ coeffs,   // [0]=center, [1..radius]=per-axis neighbor coeff
    float* __restrict__ output,
    int H, int W,     // input spatial dims (D not needed inside the kernel itself)
    int radius,
    int oD, int oH, int oW)
{
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int depth = blockIdx.z * blockDim.z + threadIdx.z;

  if (col >= oW || row >= oH || depth >= oD) return;

  int in_d = depth + radius;
  int in_r = row + radius;
  int in_c = col + radius;


  float acc = 0.0f;

  acc += input[idx3d(in_d, in_r, in_c, H, W)] * coeffs[0];
  for (int k=1; k<=radius;++k)
  {
    acc += coeffs[k] * input[idx3d(in_d + k, in_r, in_c, H, W)];
    acc += coeffs[k] * input[idx3d(in_d - k, in_r, in_c, H, W)];
    acc += coeffs[k] * input[idx3d(in_d, in_r + k, in_c, H, W)];
    acc += coeffs[k] * input[idx3d(in_d, in_r - k, in_c, H, W)];
    acc += coeffs[k] * input[idx3d(in_d, in_r, in_c + k, H, W)];
    acc += coeffs[k] * input[idx3d(in_d, in_r, in_c - k, H, W)];
  }

  output[idx3d(depth, row, col, oH, oW)] = acc;
}

torch::Tensor launch_naive(torch::Tensor input, torch::Tensor coeffs, int radius)
{
  int D = input.size(0);
  int H = input.size(1);
  int W = input.size(2);

  int oD = D - 2 * radius;
  int oH = H - 2 * radius;
  int oW = W - 2 * radius;

  // Return an empty tensor if output dimensions are non-positive
  if (oD <= 0 || oH <= 0 || oW <= 0) {
      return torch::zeros({0, 0, 0}, input.options());
  }

  torch::Tensor output = torch::zeros({oD, oH, oW}, input.options());

  dim3 blk(8,8,8);
  dim3 grd((oW + blk.x - 1) / blk.x, (oH + blk.y - 1) / blk.y, (oD + blk.z - 1) / blk.z);
  stencil_naive_kernel<<<grd, blk>>>(
    input.data_ptr<float>(),
    coeffs.data_ptr<float>(),
    output.data_ptr<float>(),
    H, W, radius, oD, oH, oW);
  return output;

}

void copyCoeffsToConstant(torch::Tensor coeffs)
{
  cudaMemcpyToSymbol(
    d_coeffs, // global constant memory
    coeffs.data_ptr<float>(),
    coeffs.numel() * sizeof(float),
    0,
    cudaMemcpyDeviceToDevice
  );
}

// ============================================================
// VERSION 2: GPU NAIVE + coeffs cached in constant memory
// One thread per output point. Reads its 6*radius+1 neighbors
// directly from global memory; coefficients cached in constmem
// (length radius+1: [0]=center, [1..radius]=taps).
// ============================================================
__global__ void stencil_constmem_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int H, int W,     // input spatial dims (D not needed inside the kernel itself)
    int radius,
    int oD, int oH, int oW)
{
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int depth = blockIdx.z * blockDim.z + threadIdx.z;

  if (col >= oW || row >= oH || depth >= oD) return;

  int in_d = depth + radius;
  int in_r = row + radius;
  int in_c = col + radius;


  float acc = 0.0f;

  acc += input[idx3d(in_d, in_r, in_c, H, W)] * d_coeffs[0];
  for (int k=1; k<=radius;++k)
  {
    acc += d_coeffs[k] * input[idx3d(in_d + k, in_r, in_c, H, W)];
    acc += d_coeffs[k] * input[idx3d(in_d - k, in_r, in_c, H, W)];
    acc += d_coeffs[k] * input[idx3d(in_d, in_r + k, in_c, H, W)];
    acc += d_coeffs[k] * input[idx3d(in_d, in_r - k, in_c, H, W)];
    acc += d_coeffs[k] * input[idx3d(in_d, in_r, in_c + k, H, W)];
    acc += d_coeffs[k] * input[idx3d(in_d, in_r, in_c - k, H, W)];
  }

  output[idx3d(depth, row, col, oH, oW)] = acc;
}

torch::Tensor launch_constmem(torch::Tensor input, int radius)
{
  int D = input.size(0);
  int H = input.size(1);
  int W = input.size(2);

  int oD = D - 2 * radius;
  int oH = H - 2 * radius;
  int oW = W - 2 * radius;

  // Return an empty tensor if output dimensions are non-positive
  if (oD <= 0 || oH <= 0 || oW <= 0) {
      return torch::zeros({0, 0, 0}, input.options());
  }

  torch::Tensor output = torch::zeros({oD, oH, oW}, input.options());

  dim3 blk(8,8,8);
  dim3 grd((oW + blk.x - 1) / blk.x, (oH + blk.y - 1) / blk.y, (oD + blk.z - 1) / blk.z);

  stencil_constmem_kernel<<<grd, blk>>>(
    input.data_ptr<float>(),
    output.data_ptr<float>(),
    H, W, radius, oD, oH, oW);
  return output;

}

#define OUT_TILE 8

// ============================================================
// VERSION 3: Shared memory tiling + Version 2
// Load input tensor from global memory to shared memory and perform stencil sweep
// Prediction : Much less effective than convolution due to more number of halo cell percentage
// and with block dim as 8,8,8, memory coalescing is affected.
// ============================================================
__global__ void stencil_smem_tiling(
    const float* __restrict__ input,
    float* __restrict__ output,
    int H, int W, int D,
    int radius, int in_tile,
    int oD, int oH, int oW)
{
  extern __shared__ float s_mem[];
  int tz = threadIdx.z;
  int ty = threadIdx.y;
  int tx = threadIdx.x;

  int col = blockIdx.x * OUT_TILE + tx;
  int row = blockIdx.y * OUT_TILE + ty;
  int depth = blockIdx.z * OUT_TILE + tz;


  int global_base_d = blockIdx.z * OUT_TILE;
  int global_base_r = blockIdx.y * OUT_TILE;
  int global_base_c = blockIdx.x * OUT_TILE;



  for (int i = tz; i < in_tile; i += OUT_TILE)
  {
    for (int j = ty; j < in_tile; j += OUT_TILE)
    {
      for (int k = tx; k < in_tile; k += OUT_TILE)
      {
        int g_d = global_base_d + i;
        int g_r = global_base_r + j;
        int g_c = global_base_c + k;

        if (g_d < D && g_r < H && g_c < W)
        {
          s_mem[smem_idx(i,j,k,in_tile)] = input[idx3d(g_d, g_r, g_c, H, W)];
        }
        else
        {
          s_mem[smem_idx(i,j,k,in_tile)] = 0.0f;
        }
      }
    }
  }

  __syncthreads();

  // Out-of-bounds check for global output writing
  if (col >= oW || row >= oH || depth >= oD) return;

  int s_d = tz + radius;
  int s_r = ty + radius;
  int s_c = tx + radius;
  float acc = s_mem[smem_idx(s_d,s_r,s_c,in_tile)] * d_coeffs[0];

  for (int k = 1; k <= radius; ++k)
  {
    acc += d_coeffs[k] * s_mem[smem_idx(s_d + k,s_r,s_c, in_tile)];
    acc += d_coeffs[k] * s_mem[smem_idx(s_d - k,s_r,s_c, in_tile)];
    acc += d_coeffs[k] * s_mem[smem_idx(s_d ,s_r + k,s_c, in_tile)];
    acc += d_coeffs[k] * s_mem[smem_idx(s_d ,s_r - k,s_c, in_tile)];
    acc += d_coeffs[k] * s_mem[smem_idx(s_d ,s_r,s_c + k, in_tile)];
    acc += d_coeffs[k] * s_mem[smem_idx(s_d ,s_r ,s_c - k, in_tile)];
  }

  output[idx3d(depth, row, col, oH, oW)] = acc;

}

torch::Tensor launch_smemtile(torch::Tensor input, int radius)
{
  int D = input.size(0);
  int H = input.size(1);
  int W = input.size(2);

  int oD = D - 2 * radius;
  int oH = H - 2 * radius;
  int oW = W - 2 * radius;

  // Return an empty tensor if output dimensions are non-positive
  if (oD <= 0 || oH <= 0 || oW <= 0) {
      return torch::zeros({0, 0, 0}, input.options());
  }

  torch::Tensor output = torch::zeros({oD, oH, oW}, input.options());

  dim3 blk(8,8,8);
  dim3 grd((oW + blk.x - 1) / blk.x, (oH + blk.y - 1) / blk.y, (oD + blk.z - 1) / blk.z);

  int in_tile = OUT_TILE + 2 * radius;

  stencil_smem_tiling<<<grd, blk, in_tile*in_tile*in_tile*sizeof(float)>>>(
    input.data_ptr<float>(),
    output.data_ptr<float>(),
    H, W, D, radius, in_tile, oD, oH, oW);
  return output;

}

#define OUT_TILE_TC 16

// ============================================================
// VERSION 4: Thread Coarsening
// Load input tensor from global memory to shared memory and perform stencil sweep
// Compute stencil sweeps for multiple x-y plane using same thread
// block Dimension : 16,16,16
// ============================================================
__global__ void stencil_thread_coarse(
    const float* __restrict__ input,
    float* __restrict__ output,
    int H, int W, int D,
    int radius, int in_tile,
    int oD, int oH, int oW)
{
  extern __shared__ float s_mem[];
  int plane_elems = in_tile * in_tile;
  int n_planes = 2 * radius + 1;

  float* planes[2 * MAXR + 1]; // allocating for max radius case
  for (int p=0;p<n_planes;++p) planes[p] = s_mem + p * plane_elems;

  int ty = threadIdx.y;
  int tx = threadIdx.x;

  int col = blockIdx.x * OUT_TILE_TC + tx;
  int row = blockIdx.y * OUT_TILE_TC + ty;

  int global_base_r = blockIdx.y * OUT_TILE_TC;
  int global_base_c = blockIdx.x * OUT_TILE_TC;

  // initial fill : z = 0 to z = 2 * radius
  int z_start = blockIdx.z * OUT_TILE_TC;
  for (int z = z_start;z < (z_start + 2 * radius + 1); ++z)
  {
    for (int i=ty;i<in_tile;i+=OUT_TILE_TC)
    {
      for (int j=tx;j<in_tile;j+=OUT_TILE_TC)
      {
        int g_r = global_base_r + i;
        int g_c = global_base_c + j;
        if (z < D && g_r < H && g_c < W)
        {
          planes[z - z_start][plane_idx(i,j,in_tile)] = input[idx3d(z,g_r, g_c, H, W)];
        }
        else
        {
          planes[z - z_start][plane_idx(i,j,in_tile)] = 0.0f;
        }
      }
    }
  }
  int s_r = ty + radius;
  int s_c = tx + radius;
  for (int zc = 0; zc < OUT_TILE_TC; ++zc)
  {
    __syncthreads();

    int z = z_start + zc;
    if (z >= oD) break;

    float acc = planes[radius][plane_idx(s_r, s_c, in_tile)] * d_coeffs[0];
    for (int k=1;k<=radius;++k)
    {
      acc += d_coeffs[k] * planes[radius][plane_idx(s_r, s_c+k, in_tile)];
      acc += d_coeffs[k] * planes[radius][plane_idx(s_r, s_c-k, in_tile)];
      acc += d_coeffs[k] * planes[radius][plane_idx(s_r+k, s_c, in_tile)];
      acc += d_coeffs[k] * planes[radius][plane_idx(s_r-k, s_c, in_tile)];
      acc += d_coeffs[k] * planes[radius-k][plane_idx(s_r, s_c, in_tile)];
      acc += d_coeffs[k] * planes[radius+k][plane_idx(s_r, s_c, in_tile)];
    }
    if (col < oW && row < oH)
      output[idx3d(z, row, col, oH, oW)] = acc;
    __syncthreads();
    float* dropped = planes[0];
    for (int p=0;p<n_planes-1;++p) planes[p] = planes[p+1];
    planes[n_planes-1] = dropped;

    int newD = z + 2*radius + 1;

    for (int i=ty;i<in_tile;i+=OUT_TILE_TC)
    {
      for (int j=tx;j<in_tile;j+=OUT_TILE_TC)
      {
        int g_r = global_base_r + i;
        int g_c = global_base_c + j;
        if (newD < D && g_r < H && g_c < W)
        {
          planes[n_planes-1][plane_idx(i,j,in_tile)] = input[idx3d(newD,g_r, g_c, H, W)];
        }
        else
        {
          planes[n_planes-1][plane_idx(i,j,in_tile)] = 0.0f;
        }
      }
    }

  }
}

torch::Tensor launch_threadcoarse(torch::Tensor input, int radius)
{
  int D = input.size(0);
  int H = input.size(1);
  int W = input.size(2);

  int oD = D - 2 * radius;
  int oH = H - 2 * radius;
  int oW = W - 2 * radius;

  // Return an empty tensor if output dimensions are non-positive
  if (oD <= 0 || oH <= 0 || oW <= 0) {
      return torch::zeros({0, 0, 0}, input.options());
  }

  torch::Tensor output = torch::zeros({oD, oH, oW}, input.options());

  dim3 blk(OUT_TILE_TC,OUT_TILE_TC);
  dim3 grd((oW + blk.x - 1) / blk.x, (oH + blk.y - 1) / blk.y, (oD + OUT_TILE_TC - 1) / OUT_TILE_TC);

  int in_tile = OUT_TILE_TC + 2 * radius;

  stencil_thread_coarse<<<grd, blk, (2*radius+1)*in_tile*in_tile*sizeof(float)>>>(
    input.data_ptr<float>(),
    output.data_ptr<float>(),
    H, W, D, radius, in_tile, oD, oH, oW);
  return output;

}

// ============================================================
// VERSION 5: Register Tiling
//
// Only the x/y in-plane neighbors need a halo (they read
// neighboring columns/rows). The z neighbors always read the
// exact same (row, col) as the center — no halo needed for
// them at all. So: keep ONE haloed x-y plane in shared memory
// (the current z-center), and keep the z-window as a small
// per-thread array that (thanks to templating on RADIUS below)
// the compiler can fully unroll and allocate in registers,
// instead of a shared/synchronized buffer.
//
// Templated on RADIUS so `zreg`'s size and the tap loops are
// compile-time constants — a runtime `radius` loop bound would
// force zreg into local memory (global-memory-backed), defeating
// the point of "register" tiling.
// ============================================================

#define Z_CHUNK OUT_TILE_TC   // z-depth each block sweeps; independent knob from the x/y tile size — tune separately

template <int RADIUS>
__global__ void stencil_register_tile(
    const float* __restrict__ input,
    float* __restrict__ output,
    int H, int W, int D,
    int in_tile,     // OUT_TILE_TC + 2*RADIUS, x/y halo tile width
    int oD, int oH, int oW)
{
    extern __shared__ float s_mem[];     // ONE haloed x-y plane — the current z-center only
    float zreg[2 * RADIUS + 1];          // per-thread z-window; compile-time sized -> registers

    int ty = threadIdx.y;
    int tx = threadIdx.x;

    int col = blockIdx.x * OUT_TILE_TC + tx;   // output-space column
    int row = blockIdx.y * OUT_TILE_TC + ty;   // output-space row

    // input-space position for THIS thread's own column — the z-taps always
    // read here, never offset, since the stencil is an axis-aligned cross.
    int in_row = row + RADIUS;
    int in_col = col + RADIUS;

    int global_base_r = blockIdx.y * OUT_TILE_TC;
    int global_base_c = blockIdx.x * OUT_TILE_TC;

    int z_start = blockIdx.z * Z_CHUNK;   // first output depth this block owns

    // ---- initial fill: z-window covering input depths [z_start, z_start+2*RADIUS] ----
    // zreg[RADIUS] ends up holding the center depth for output z = z_start.
    #pragma unroll
    for (int p = 0; p < 2 * RADIUS + 1; ++p) {
        int z = z_start + p;
        zreg[p] = (z < D && in_row < H && in_col < W)
                      ? input[idx3d(z, in_row, in_col, H, W)]
                      : 0.0f;
    }

    // ---- initial fill: haloed x-y tile for the center plane (z_start+RADIUS) ----
    for (int i = ty; i < in_tile; i += OUT_TILE_TC) {
        for (int j = tx; j < in_tile; j += OUT_TILE_TC) {
            int g_r = global_base_r + i;
            int g_c = global_base_c + j;
            s_mem[plane_idx(i, j, in_tile)] =
                (z_start + RADIUS < D && g_r < H && g_c < W)
                    ? input[idx3d(z_start + RADIUS, g_r, g_c, H, W)]
                    : 0.0f;
        }
    }

    int s_r = ty + RADIUS;   // this thread's row inside the shared tile
    int s_c = tx + RADIUS;   // this thread's col inside the shared tile

    // ---- sweep this block's z-chunk ----
    for (int zc = 0; zc < Z_CHUNK; ++zc) {
        __syncthreads();   // s_mem (and zreg, on the first pass) must be fully loaded first

        int z = z_start + zc;
        if (z >= oD) break;   // same for every thread in the block -> safe, no barrier divergence

        // center tap
        float acc = s_mem[plane_idx(s_r, s_c, in_tile)] * d_coeffs[0];

        #pragma unroll
        for (int k = 1; k <= RADIUS; ++k) {
            acc += d_coeffs[k] * s_mem[plane_idx(s_r, s_c + k, in_tile)];  // +x, shared
            acc += d_coeffs[k] * s_mem[plane_idx(s_r, s_c - k, in_tile)];  // -x, shared
            acc += d_coeffs[k] * s_mem[plane_idx(s_r + k, s_c, in_tile)];  // +y, shared
            acc += d_coeffs[k] * s_mem[plane_idx(s_r - k, s_c, in_tile)];  // -y, shared
            acc += d_coeffs[k] * zreg[RADIUS - k];                        // -z, register
            acc += d_coeffs[k] * zreg[RADIUS + k];                        // +z, register
        }

        if (col < oW && row < oH)
            output[idx3d(z, row, col, oH, oW)] = acc;

        __syncthreads();   // everyone must finish reading s_mem before it gets overwritten below

        // ---- slide the z-window by one: drop oldest, append the new leading edge ----
        #pragma unroll
        for (int p = 0; p < 2 * RADIUS; ++p) zreg[p] = zreg[p + 1];

        int new_z_tap = z + 2 * RADIUS + 1;   // one past the window's previous far edge
        zreg[2 * RADIUS] = (new_z_tap < D && in_row < H && in_col < W)
                                ? input[idx3d(new_z_tap, in_row, in_col, H, W)]
                                : 0.0f;

        // ---- slide the shared tile forward by exactly one plane ----
        int new_center = z + RADIUS + 1;
        for (int i = ty; i < in_tile; i += OUT_TILE_TC) {
            for (int j = tx; j < in_tile; j += OUT_TILE_TC) {
                int g_r = global_base_r + i;
                int g_c = global_base_c + j;
                s_mem[plane_idx(i, j, in_tile)] =
                    (new_center < D && g_r < H && g_c < W)
                        ? input[idx3d(new_center, g_r, g_c, H, W)]
                        : 0.0f;
            }
        }
    }
}

// ============================================================
// WRAPPER — dispatches to the matching template instantiation
// since `radius` only exists at runtime here but the kernel
// needs it at compile time.
// ============================================================
torch::Tensor launch_registertile(torch::Tensor input, int radius)
{
    int D = input.size(0);
    int H = input.size(1);
    int W = input.size(2);

    int oD = D - 2 * radius;
    int oH = H - 2 * radius;
    int oW = W - 2 * radius;

    if (oD <= 0 || oH <= 0 || oW <= 0)
        return torch::zeros({0, 0, 0}, input.options());

    torch::Tensor output = torch::zeros({oD, oH, oW}, input.options());

    int in_tile = OUT_TILE_TC + 2 * radius;
    size_t smem_bytes = (size_t)in_tile * in_tile * sizeof(float);   // ONE plane, not (2r+1)

    dim3 blk(OUT_TILE_TC, OUT_TILE_TC);
    dim3 grd(
        (oW + OUT_TILE_TC - 1) / OUT_TILE_TC,
        (oH + OUT_TILE_TC - 1) / OUT_TILE_TC,
        (oD + Z_CHUNK - 1) / Z_CHUNK);

    switch (radius) {
        case 1:
            stencil_register_tile<1><<<grd, blk, smem_bytes>>>(
                input.data_ptr<float>(), output.data_ptr<float>(), H, W, D, in_tile, oD, oH, oW);
            break;
        case 2:
            stencil_register_tile<2><<<grd, blk, smem_bytes>>>(
                input.data_ptr<float>(), output.data_ptr<float>(), H, W, D, in_tile, oD, oH, oW);
            break;
        case 3:
            stencil_register_tile<3><<<grd, blk, smem_bytes>>>(
                input.data_ptr<float>(), output.data_ptr<float>(), H, W, D, in_tile, oD, oH, oW);
            break;
        default:
            TORCH_CHECK(false, "launch_registertile: radius must be 1..", MAXR);
    }

    return output;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("launch_naive", &launch_naive, "naive 3D stencil (global memory)");
    m.def("copyCoeffsToConstant", &copyCoeffsToConstant, "copy coeffs to constant memory");
    m.def("launch_constmem", &launch_constmem, "naive 3D stencil (global memory), constant coeffs");
    m.def("launch_smemtile", &launch_smemtile, "shared memory tiling");
    m.def("launch_threadcoarse",&launch_threadcoarse, "thread coarsening");
    m.def("launch_registertile", &launch_registertile, "register tiling");
}


Overwriting stencil_kernels.cu


In [ ]:
# ------------------------------------------------------------
# NOTE: this cell's env-var setup + imports are repeated (and
# actually used) in the next cell, which also calls load(). Kept
# here as-is since it doesn't hurt anything, but the next cell is
# the one that matters for compilation.
# ------------------------------------------------------------
import os

os.environ["CUDA_HOME"] = "/usr/local/cuda"          # points nvcc's toolchain lookup at Colab's CUDA install
os.environ["TORCH_SHOW_CPP_EXTENSION_DEBUG"] = "1"    # prints the full nvcc/g++ command line on build — useful when compile errors are cryptic

import torch
import pandas as pd            # only used for the pivot tables in benchmark_sweep()
from torch.utils.cpp_extension import load   # JIT-compiles stencil_kernels.cu into a Python-importable module

In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'   # nvcc toolchain path
import torch
from torch.utils.cpp_extension import load

os.environ['TORCH_SHOW_CPP_EXTENSION_DEBUG'] = '1'   # verbose nvcc/g++ output on build

# -gencode targets Turing (T4's architecture, CC 7.5) specifically. If this
# notebook is ever run on a different GPU (e.g. Kaggle's P100, CC 6.0), this
# flag needs to change to match, or the compiled kernels will fail to load.
extra_cuda_cflags = ['-gencode=arch=compute_75,code=sm_75']
extra_ldflags = ['-lcudart']   # link CUDA runtime explicitly (sometimes needed under JIT compilation)

# Keeping the build in an explicit, inspectable directory (rather than the
# default hidden cache dir) makes it easy to check generated .so files /
# ninja logs directly if something goes wrong.
build_dir = os.path.join(os.getcwd(), 'cuda_build')
os.makedirs(build_dir, exist_ok=True)

# ------------------------------------------------------------
# KNOWN GOTCHA when iterating on stencil_kernels.cu: CPython cannot
# reload a native extension module under the same name within one
# live Python process. Re-running this cell after editing the .cu
# file (same `name="stencil_kernels"`) raises:
#   ImportError: dynamic module does not define module export
#   function (PyInit_stencil_kernels)
# This is NOT a compile error — check the verbose nvcc output above
# it first. If the .cu itself compiled cleanly, fix by either:
#   (a) Runtime -> Restart session, then re-run this cell, or
#   (b) give `name=` a fresh value each rebuild (e.g. append
#       int(time.time())) so no restart is needed, or
#   (c) `!rm -rf cuda_build` if a stale cached .so is being reused.
# ------------------------------------------------------------
stencil_module = load(
      name     = "stencil_kernels",
      sources  = ["/content/stencil_kernels.cu"],
      extra_cuda_cflags = extra_cuda_cflags,
      extra_ldflags = extra_ldflags,
      build_directory = build_dir, # Use the custom build directory
      verbose  = True
  )

In [ ]:
# Every KERNELS entry (below) is called as fn(inp, coeffs, radius) — one
# common interface — even though only run_naive actually uses `coeffs` as
# an argument. The other four kernels read their coefficients from GPU
# __constant__ memory instead (copied there beforehand by KERNEL_SETUP),
# so `coeffs` is accepted here purely to keep the call signature uniform
# across all five kernel versions.

def run_naive(inp, coeffs, radius):
    return stencil_module.launch_naive(inp, coeffs, radius)

def run_constmem(inp, coeffs, radius):
    return stencil_module.launch_constmem(inp, radius)

def run_smemtile(inp, coeffs, radius):
    return stencil_module.launch_smemtile(inp, radius)

def run_threadcoarse(inp, coeffs, radius):
  return stencil_module.launch_threadcoarse(inp, radius)

def run_registertile(inp, coeffs, radius):
  return stencil_module.launch_registertile(inp, radius)

In [ ]:
# name -> callable. Every entry must accept (inp, coeffs, radius) and
# return the output tensor — this is what lets check_all()/benchmark_all()
# below loop over all five versions identically.
#
# NAMING NOTE: "4_register_tiled" is chronologically/conceptually Version 5
# (see problem.md "Versions to Implement"). Thread coarsening (V4) was built
# and benchmarked FIRST as the initial attempt at the z-sweep; register
# tiling (V5) came after, once V4 turned out to underperform even naive.
# The "4_" prefix here is a harness-naming leftover, not a real version
# number — see notes.md for the actual build order and why V5 was needed
# on top of V4.
KERNELS = {
    "1_naive": run_naive,
    "2_constmem":       run_constmem,
    "3_shared_tiled":   run_smemtile,
    "4_thread_coarse": run_threadcoarse,
     "4_register_tiled": run_registertile,
}

# Kernels that read coefficients from __constant__ memory (everything
# except naive) need this called once before every timed/checked call —
# NOT just once overall — since coefficients differ across radii and the
# harness reuses one compiled module across the whole radius sweep.
KERNEL_SETUP = {
    "2_constmem": lambda inp, coeffs, radius:
         stencil_module.copyCoeffsToConstant(coeffs),
    "3_shared_tiled": lambda inp, coeffs, radius:
         stencil_module.copyCoeffsToConstant(coeffs),
    "4_thread_coarse": lambda inp, coeffs, radius:
         stencil_module.copyCoeffsToConstant(coeffs),
     "4_register_tiled" : lambda inp, coeffs, radius:
         stencil_module.copyCoeffsToConstant(coeffs),
}

In [ ]:
# ------------------------------------------------------------
# COMPARER — correctness only. Every kernel version is checked against
# run_torch_reference() (F.conv3d with a dense, zero-padded kernel — see
# problem.md Concept 6 for why this is a correctness oracle only, never a
# performance target: it pays for the full dense (2r+1)^3 cube while our
# kernels only ever touch the 6r+1 nonzero taps).
# ------------------------------------------------------------
def check_version(name, fn, inp, coeffs, radius, atol=1e-4):
    if name in KERNEL_SETUP:
        KERNEL_SETUP[name](inp, coeffs, radius)   # push coeffs to constant memory first, if this version needs it

    out = fn(inp, coeffs, radius)
    expected = run_torch_reference(inp, radius)

    if out.shape != expected.shape:
        print(
            f"  [{name}] FAIL — shape mismatch: "
            f"got {tuple(out.shape)}, expected {tuple(expected.shape)}"
        )
        return False

    max_err = (out - expected).abs().max().item()
    ok = torch.allclose(out, expected, atol=atol)
    print(f"  [{name}] {'PASS' if ok else 'FAIL'}  max_err={max_err:.3e}")
    return ok


def check_all(N=64, radius=1, atol=1e-4):
    """Runs every registered kernel once against the reference for one
    (N, radius) combination and prints PASS/FAIL for each."""
    print(f"N={N}  radius={radius}")
    inp = torch.randn(N, N, N, device="cuda", dtype=torch.float32).contiguous()
    coeffs = build_coeffs_array(radius)

    results = {}
    for name, fn in KERNELS.items():
        results[name] = check_version(name, fn, inp, coeffs, radius, atol=atol)
    return results

In [ ]:
# ------------------------------------------------------------
# BENCHMARK — the actual "how do our kernels stack up against each
# other" comparison (problem.md Q2-Q6). Timing and bandwidth/GFLOPS are
# computed on the KERNELS entries ONLY — never on run_torch_reference(),
# since F.conv3d does the full dense (2r+1)^3 cube under the hood and
# would make bandwidth/GFLOPS numbers meaningless for our sparse
# 6r+1-tap kernels (see problem.md Concept 6).
# ------------------------------------------------------------
def benchmark_version(name, fn, inp, coeffs, radius, N, warmup=3, runs=20):
    if name in KERNEL_SETUP:
        KERNEL_SETUP[name](inp, coeffs, radius)   # one-time, untimed — not part of the measured loop

    def call():
        return fn(inp, coeffs, radius)

    time_ms = gpu_timer(call, warmup=warmup, runs=runs)
    metrics = compute_metrics(N, radius, time_ms)
    metrics["version"] = name
    print(
        f"  [{name}] time={time_ms:8.4f} ms  "
        f"BW={metrics['bandwidth_GBs']:7.1f} GB/s "
        f"({metrics['pct_peak_bw']:4.1f}% peak)  "
        f"{metrics['gflops']:7.1f} GFLOPS"
    )
    return metrics

In [ ]:
def benchmark_all(N, radius, warmup=3, runs=20):
    """Times every registered kernel once for one (N, radius) pair, on a
    SHARED random input tensor (so every version sees identical data for
    that combination), and returns their metrics rows."""
    print(f"N={N}  radius={radius}")
    inp = torch.randn(N, N, N, device="cuda", dtype=torch.float32).contiguous()
    coeffs = build_coeffs_array(radius)

    rows = []
    for name, fn in KERNELS.items():
        rows.append(benchmark_version(name, fn, inp, coeffs, radius, N, warmup, runs))
    return rows


def benchmark_sweep(n_values, r_values, warmup=3, runs=20):
    """Runs benchmark_all() over every (N, radius) combination, then
    pivots the results into one 'version x N' table per radius — the
    'how do our kernels stack up against each other' view. These pivot
    tables are what get copied into notes.md's Results section."""
    rows = []
    for N in n_values:
        for radius in r_values:
            rows.extend(benchmark_all(N, radius, warmup, runs))
            print()

    df = pd.DataFrame(rows)
    for radius in r_values:
        sub = df[df.radius == radius]
        print("\n" + "=" * 68)
        print(f"  Kernel Time (ms)  —  radius={radius}")
        print("=" * 68)
        print(sub.pivot(index="version", columns="N", values="time_ms").to_string())

        print(f"\n  Effective Bandwidth (GB/s)  —  radius={radius}  [T4 peak = 320]")
        print(sub.pivot(index="version", columns="N", values="bandwidth_GBs").to_string())

    return df

In [ ]:
def correctness_sweep(n_values, r_values):
    """Runs check_all() over every (N, radius) combination and returns
    True only if every registered kernel passed every check. Gate this
    before trusting benchmark_sweep()'s timings — a kernel that's fast
    because it's wrong isn't a useful result."""
    all_ok = True
    for N in n_values:
        for radius in r_values:
            results = check_all(N=N, radius=radius)
            all_ok = all_ok and all(results.values())
            print()
    return all_ok

In [ ]:
# CORRECTNESS_N: kept small (and includes a size not in BENCHMARK_N) so
# correctness checks stay fast, while still independently exercising the
# boundary arithmetic at a size the benchmark sweep doesn't happen to hit.
CORRECTNESS_N = [32, 64, 128]

# BENCHMARK_N: matches problem.md "Input Sizes to Test" — chosen to stay
# well within the T4's 15.6 GB while covering roughly two orders of
# magnitude of input volume (64^3 = 1 MB up to 384^3 = 226 MB).
BENCHMARK_N = [64, 128, 256, 384]

# Radius sweep = stencil order: 1 -> 2nd order (7-point), 2 -> 4th order
# (13-point), 3 -> 6th order (19-point). See problem.md Concept 1.
R_VALUES = [1, 2, 3]

In [ ]:
# Gate: only run the (much longer) benchmark sweep below if every kernel
# version agrees with the torch reference at every tested size/radius.
print("=" * 68)
print("CORRECTNESS")
print("=" * 68)
all_ok = correctness_sweep(CORRECTNESS_N, R_VALUES)
print("ALL PASS" if all_ok else "SOME FAILED — see above")

In [ ]:
if all_ok:
  print("\n" + "=" * 68)
  print("BENCHMARK")
  print("=" * 68)
  benchmark_sweep(BENCHMARK_N, R_VALUES)   # prints the per-radius pivot tables that went into notes.md
else:
    print("\nSkipping benchmark sweep — fix correctness failures first.")